### Calculating the QM solution of the reaction between $OH^-$ and $CH_3Cl$

In [26]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyscf import gto, scf
from pyscf.geomopt.geometric_solver import optimize
from pyscf.qmmm import mm_charge
from scipy.optimize import minimize
from scipy.spatial.transform import Rotation
import pickle

In [27]:
au_to_kJ_conversion = 2625.49962
R = 8.314462618  # J/(mol K)
SCF_CONV_TOL = 1e-6

In [28]:
def move_oh_to_distance(coords, target_distance):

    coords = coords.copy()

    O = 0
    H_oh = 1
    C = 2

    carbon = coords[C]
    oxygen = coords[O]

    # Current direction from C toward O
    direction = oxygen - carbon
    direction /= np.linalg.norm(direction)

    # Preserve the O-H vector
    oh_vector = coords[H_oh] - coords[O]

    # Place O at the desired C-O distance
    new_oxygen = carbon + direction * target_distance

    # Move H together with O
    new_hydrogen = new_oxygen + oh_vector

    coords[O] = new_oxygen
    coords[H_oh] = new_hydrogen

    return coords

def build_molecule(coords):

    atom_string = ""

    for symbol, coord in zip(symbols, coords):

        x, y, z = coord

        atom_string += (
            f"{symbol} "
            f"{x:.10f} "
            f"{y:.10f} "
            f"{z:.10f}\n"
        )

    return gto.M(
        atom=atom_string,
        basis="6-31G",
        charge=-1,
        spin=0,
        unit="Angstrom"
    )
    
def write_constraint(distance):

    with open("constraints.txt", "w") as f:

        f.write("$set\n")
        f.write(
            f"distance 1 3 {distance:.8f}\n"
        )

def make_scf_with_solvent(mol):

    mf = scf.RHF(mol)
    mf.conv_tol = SCF_CONV_TOL

    # Add implicit solvent using ddCOSMO
    # mf = mf.ddCOSMO()

    # Water dielectric constant
    # mf.with_solvent.eps = 78.3553

    return mf
    
def optimize_at_distance(
    mol,
    distance,
    water_x,
    maxsteps=25
):

    # ------------------------------------------------
    # 1. Build the current TIP4P-D water
    # ------------------------------------------------

    O, H1, H2, M = water_from_variables(water_x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------
    # 2. Create the C-O constraint
    # ------------------------------------------------

    write_constraint(distance)

    # ------------------------------------------------
    # 3. Build QM/MM SCF object
    # ------------------------------------------------

    mf = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf.conv_tol = SCF_CONV_TOL

    # ------------------------------------------------
    # 4. Create combined QM/MM + LJ gradient
    # ------------------------------------------------

    grad = QMMM_LJ_Gradients(
        mf,
        O,
        qm_atom_types
    )

    # ------------------------------------------------
    # 5. Geometry optimization
    # ------------------------------------------------

    mol_opt = optimize(
        grad,
        constraints="constraints.txt",
        maxsteps=maxsteps
    )

    # ------------------------------------------------
    # 6. Bare QM energy
    # ------------------------------------------------

    mf_qm = scf.RHF(mol_opt)
    mf_qm.conv_tol = SCF_CONV_TOL

    energy_qm = mf_qm.kernel()

    # ------------------------------------------------
    # 7. Final QM/MM energy
    # ------------------------------------------------

    mf_qmmm_final = mm_charge(
        scf.RHF(mol_opt),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )

    mf_qmmm_final.conv_tol = SCF_CONV_TOL

    energy_qmmm = mf_qmmm_final.kernel()

    # ------------------------------------------------
    # 8. Return
    # ------------------------------------------------

    return (
        mol_opt,
        energy_qm,
        energy_qmmm
    )

def write_xyz(mol, filename, distance):

    coords = mol.atom_coords(
        unit="Angstrom"
    )

    with open(filename, "w") as f:

        f.write(f"{mol.natm}\n")

        f.write(
            f"C-O distance = {distance:.8f} Angstrom\n"
        )

        for symbol, coord in zip(symbols, coords):

            x, y, z = coord

            f.write(
                f"{symbol:2s} "
                f"{x:14.8f} "
                f"{y:14.8f} "
                f"{z:14.8f}\n"
            )

def tip4pd_electrostatic_potential(qm_coords, water_positions):
    """
    Electrostatic potential at each QM atom due to one TIP4P-D water.

    Coordinates are in Angstrom.
    Charges are in units of e.

    Returns
    -------
    potential : ndarray
        Electrostatic potential at each QM atom in atomic units (Hartree/e).
    """

    # 1 Angstrom = 1.889726125 bohr
    ANGSTROM_TO_BOHR = 1.889726125

    qm_bohr = np.asarray(qm_coords) * ANGSTROM_TO_BOHR
    water_bohr = np.asarray(water_positions) * ANGSTROM_TO_BOHR

    potential = np.zeros(len(qm_bohr))

    # Coulomb constant in atomic units is 1
    for q, r_water in zip(TIP4P_D_CHARGES, water_bohr):

        if q == 0.0:
            continue

        distances = np.linalg.norm(qm_bohr - r_water, axis=1)

        potential += q / distances

    return potential

def lj_energy(r_A, sigma_A, epsilon_kJmol):
    """
    Lennard-Jones 12-6 energy.

    Parameters
    ----------
    r_A : float
        Distance in Angstrom
    sigma_A : float
        LJ sigma in Angstrom
    epsilon_kJmol : float
        LJ epsilon in kJ/mol

    Returns
    -------
    float
        LJ energy in kJ/mol
    """
    sr6 = (sigma_A / r_A)**6
    return 4.0 * epsilon_kJmol * (sr6**2 - sr6)

def qm_water_lj_energy(qm_coords_A, water_O_coords_A, qm_atom_types):
    water_O = np.asarray(water_O_coords_A[0])

    E_LJ = 0.0

    for i, atom_type in enumerate(qm_atom_types):

        # Skip atoms for which we have no LJ parameters
        if atom_type not in qm_lj:
            continue

        sigma = qm_lj[atom_type]["sigma_A"]
        epsilon = qm_lj[atom_type]["epsilon_kJmol"]

        r = np.linalg.norm(
            qm_coords_A[i] - water_O
        )

        E_LJ += lj_energy(
            r,
            sigma,
            epsilon
        )

    return E_LJ

def water_from_variables(x):
    """
    x[:3] = O position in Angstrom
    x[3:6] = rotation vector in radians

    Returns:
        O, H1, H2, M
    """

    translation = np.asarray(x[:3])
    rotation = Rotation.from_rotvec(x[3:6])

    O  = translation + rotation.apply(water_O_local)
    H1 = translation + rotation.apply(water_H1_local)
    H2 = translation + rotation.apply(water_H2_local)
    M  = translation + rotation.apply(water_M_local)

    return O, H1, H2, M

def total_qm_water_energy(x, mol, qm_atom_types, energy_qm):

    # ------------------------------------------------
    # Safety check on water translation
    # ------------------------------------------------

    if np.max(np.abs(x[:3])) > 10.0:
        return 1.0e10

    # Build water
    O, H1, H2, M = water_from_variables(x)

    # ------------------------------------------------
    # TIP4P-D electrostatic sites
    # ------------------------------------------------

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------
    # QM/MM electrostatics
    # ------------------------------------------------

    mf_qmmm = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )
    mf_qmmm.conv_tol = SCF_CONV_TOL
    energy_qmmm = mf_qmmm.kernel()

    # ------------------------------------------------
    # QM-water LJ interaction
    # ------------------------------------------------

    E_LJ = qm_water_lj_energy(
        mol.atom_coords(unit="Angstrom"),
        np.array([O]),
        qm_atom_types
    )

    # ------------------------------------------------
    # Electrostatic interaction
    # ------------------------------------------------

    E_electrostatic = (
        energy_qmmm - energy_qm
    ) * au_to_kJ_conversion

    return E_electrostatic + E_LJ
    
def total_qm_water_energy_full(
    x,
    mol,
    qm_atom_types
):
    """
    Full QM + one TIP4P-D water energy.

    x:
        [Ox, Oy, Oz, rx, ry, rz]

    Returns:
        E_QM + E_QM/water electrostatics + E_LJ
        in kJ/mol
    """

    # ------------------------------------------------
    # Build water
    # ------------------------------------------------

    O, H1, H2, M = water_from_variables(x)

    mm_coords = np.array([
        H1,
        H2,
        M
    ])

    mm_charges = np.array([
        0.58,
        0.58,
        -1.16
    ])

    # ------------------------------------------------
    # QM in the electrostatic field of water
    # ------------------------------------------------

    mf_qmmm = mm_charge(
        scf.RHF(mol),
        mm_coords,
        mm_charges,
        unit="Angstrom"
    )
    mf_qmmm.conv_tol = SCF_CONV_TOL
    energy_qmmm = mf_qmmm.kernel()

    # ------------------------------------------------
    # QM-water LJ
    # ------------------------------------------------

    E_LJ = qm_water_lj_energy(
        mol.atom_coords(unit="Angstrom"),
        np.array([O]),
        qm_atom_types
    )

    # ------------------------------------------------
    # Convert QM/MM energy to kJ/mol
    # ------------------------------------------------

    E_QMMM = energy_qmmm * 2625.49962

    # ------------------------------------------------
    # Total energy
    # ------------------------------------------------

    E_total = E_QMMM + E_LJ

    return E_total

def qm_water_lj_energy_gradient(
    qm_coords_A,
    water_O_A,
    qm_atom_types
):
    """
    Calculate QM-water LJ energy and gradient.

    Parameters
    ----------
    qm_coords_A : (N,3) array
        QM atom coordinates in Angstrom.

    water_O_A : (3,) array
        Water oxygen position in Angstrom.

    qm_atom_types : list
        LJ type for each QM atom.

    Returns
    -------
    E_LJ : float
        LJ energy in kJ/mol.

    grad : (N,3) array
        Gradient dE/dR in kJ/mol/Angstrom.
    """

    grad = np.zeros_like(qm_coords_A, dtype=float)
    E_LJ = 0.0

    for i, atom_type in enumerate(qm_atom_types):

        sigma = qm_lj[atom_type]["sigma_A"]
        epsilon = qm_lj[atom_type]["epsilon_kJmol"]

        r_vec = qm_coords_A[i] - water_O_A
        r = np.linalg.norm(r_vec)

        # Avoid division by zero
        if r < 1e-8:
            raise ValueError(
                f"QM atom {i} is essentially on top of water O"
            )

        sr6 = (sigma / r)**6
        sr12 = sr6**2

        # LJ energy
        E = 4.0 * epsilon * (sr12 - sr6)

        E_LJ += E

        # dE/dr
        dE_dr = (
            4.0 * epsilon / r
            * (-12.0 * sr12 + 6.0 * sr6)
        )

        # Cartesian gradient
        grad[i] = dE_dr * (r_vec / r)

    return E_LJ, grad

def qm_water_gradient(mf_qmmm, water_O_A, qm_atom_types):
    """
    QM/MM gradient + QM-water LJ gradient.

    Returns gradient in Hartree/Bohr.
    """

    # Standard PySCF QM/MM nuclear gradient
    grad_qmmm = mf_qmmm.nuc_grad_method().kernel()

    # QM coordinates in Angstrom
    qm_coords_A = mf_qmmm.mol.atom_coords(
        unit="Angstrom"
    )

    # LJ gradient in kJ/mol/Angstrom
    E_LJ, grad_LJ = qm_water_lj_energy_gradient(
        qm_coords_A,
        water_O_A,
        qm_atom_types
    )

    # Convert kJ/mol/Angstrom -> Hartree/Bohr
    grad_LJ_Ha_Bohr = (
        grad_LJ
        / 2625.49962
        / 1.889726125
    )

    # Add LJ force/gradient
    grad_total = grad_qmmm + grad_LJ_Ha_Bohr

    return grad_total

class QMMM_LJ_Gradients:
    """
    QM/MM nuclear gradient + QM-water LJ gradient.

    Gradients are returned in Hartree/Bohr.
    """

    def __init__(self, mf, water_O_A, qm_atom_types):

        self.mf = mf
        self.water_O_A = np.asarray(water_O_A)
        self.qm_atom_types = qm_atom_types

        self.mol = mf.mol

        # PySCF-style attributes expected by geomeTRIC
        self.verbose = mf.verbose
        self.stdout = mf.stdout
        self.max_memory = mf.max_memory

    # ------------------------------------------------
    # PySCF interface
    # ------------------------------------------------

    def nuc_grad_method(self):
        return self

    def as_scanner(self):
        return self

    # ------------------------------------------------
    # Gradient
    # ------------------------------------------------

    def kernel(self, *args, **kwargs):

        # --------------------------------------------
        # QM/MM gradient
        # --------------------------------------------

        grad_qmmm = self.mf.nuc_grad_method().kernel(
            *args,
            **kwargs
        )

        # --------------------------------------------
        # LJ gradient
        # --------------------------------------------

        qm_coords_A = self.mol.atom_coords(
            unit="Angstrom"
        )

        E_LJ, grad_LJ = qm_water_lj_energy_gradient(
            qm_coords_A,
            self.water_O_A,
            self.qm_atom_types
        )

        # --------------------------------------------
        # kJ/mol/Angstrom
        # ->
        # Hartree/Bohr
        # --------------------------------------------

        grad_LJ_Ha_Bohr = (
            grad_LJ
            / 2625.49962
            / 1.889726125
        )

        # --------------------------------------------
        # Combined gradient
        # --------------------------------------------

        grad_total = (
            grad_qmmm
            + grad_LJ_Ha_Bohr
        )

        return grad_total

In [29]:
initial_xyz = """
O   -5.000   0.000   0.000
H   -5.970   0.000   0.000

C    0.000   0.000   0.000
Cl   1.780   0.000   0.000

H   -0.630   0.630   0.630
H   -0.630  -0.630   0.630
H   -0.630   0.000  -0.890
"""

In [30]:
# ------------------------------------------------------------
# Classical TIP4P-D water
# ------------------------------------------------------------

OH = 0.9572       # Angstrom
HH = 1.5139       # Angstrom

# Put O at the origin.
O = np.array([0.0, 0.0, 3.0])

# Put H1 along +x.
H1 = O + np.array([OH, 0.0, 0.0])

# Construct H2 so that:
# |O-H2| = OH
# |H1-H2| = HH

cos_theta = (2 * OH**2 - HH**2) / (2 * OH**2)
sin_theta = np.sqrt(1.0 - cos_theta**2)

H2 = O + np.array([
    OH * cos_theta,
    OH * sin_theta,
    0.0
])

water_coords = np.array([O, H1, H2])

# TIP4P-D charges
water_charges = np.array([
    0.00,     # O
    +0.58,    # H1
    +0.58     # H2
])
water_m_charge = -1.16

# TIP4P-D virtual-site position
M_COEFF = 0.131937768
M = O + M_COEFF * (H1 - O) + M_COEFF * (H2 - O)

print("O :", O)
print("H1:", H1)
print("H2:", H2)
print("M :", M)

print("\nDistances:")
print("O-H1 =", np.linalg.norm(H1 - O))
print("O-H2 =", np.linalg.norm(H2 - O))
print("H1-H2 =", np.linalg.norm(H2 - H1))

O : [0. 0. 3.]
H1: [0.9572 0.     3.    ]
H2: [-0.23998617  0.92662747  3.        ]
M : [0.09462759 0.12225716 3.        ]

Distances:
O-H1 = 0.9572
O-H2 = 0.9572
H1-H2 = 1.5139


In [31]:
# ------------------------------------------------------------
# TIP4P-D electrostatic interaction with the QM region
# ------------------------------------------------------------

# TIP4P-D charges
TIP4P_D_CHARGES = np.array([
    0.00,    # O
    +0.58,   # H1
    +0.58,   # H2
    -1.16    # M
])

TIP4P_D_POSITIONS = np.vstack([
    O,
    H1,
    H2,
    M
])

In [32]:
# Published OH- LJ parameters
# de Lucas et al., J. Phys. Chem. Lett. 2024
#
# sigma in Angstrom
# epsilon in K (epsilon/kB)

oh_lj = {
    "O": {
        "sigma_A": 3.4000,
        "epsilon_K": 30.1753,
    },
    "H": {
        "sigma_A": 1.4430,
        "epsilon_K": 22.1192,
    },
}

water_lj = {
    "O": {
        "sigma_A": 3.16499897053924,
        "epsilon_kJmol": 0.936553854112664,
    }
}

print(oh_lj)
print(water_lj)


for atom in oh_lj:
    oh_lj[atom]["epsilon_kJmol"] = (
        oh_lj[atom]["epsilon_K"] * R / 1000
    )

print(oh_lj)

# Published OH- / water-O cross LJ parameters
# de Lucas et al., J. Phys. Chem. Lett. 2024
#
# sigma in Angstrom
# epsilon/kB in K

oh_water_cross = {
    "O": {
        "sigma_A": 3.4500,
        "epsilon_K": 53.0312,
    },
    "H": {
        "sigma_A": 2.30095,
        "epsilon_K": 45.4040,
    },
}

# Convert epsilon/kB from K to kJ/mol
for atom in oh_water_cross:
    oh_water_cross[atom]["epsilon_kJmol"] = (
        oh_water_cross[atom]["epsilon_K"] * R / 1000
    )

print(oh_water_cross)

{'O': {'sigma_A': 3.4, 'epsilon_K': 30.1753}, 'H': {'sigma_A': 1.443, 'epsilon_K': 22.1192}}
{'O': {'sigma_A': 3.16499897053924, 'epsilon_kJmol': 0.936553854112664}}
{'O': {'sigma_A': 3.4, 'epsilon_K': 30.1753, 'epsilon_kJmol': 0.2508914038369354}, 'H': {'sigma_A': 1.443, 'epsilon_K': 22.1192, 'epsilon_kJmol': 0.18390926154006562}}
{'O': {'sigma_A': 3.45, 'epsilon_K': 53.0312, 'epsilon_kJmol': 0.44092592998768165}, 'H': {'sigma_A': 2.30095, 'epsilon_K': 45.404, 'epsilon_kJmol': 0.377509860707672}}


In [33]:
# Get the QM coordinates directly from initial_xyz
qm_coords = np.array([
    [-5.000,  0.000,  0.000],   # O
    [-5.970,  0.000,  0.000],   # H
    [ 0.000,  0.000,  0.000],   # C
    [ 1.780,  0.000,  0.000],   # Cl
    [-0.630,  0.630,  0.630],   # H
    [-0.630, -0.630,  0.630],   # H
    [-0.630,  0.000, -0.890],   # H
])

# Evaluate the electrostatic potential energy between the QM coordinates and the water
potential = tip4pd_electrostatic_potential(
    qm_coords,
    TIP4P_D_POSITIONS
)
print("Electrostatic potential at QM atoms:")
for i, value in enumerate(potential):
    print(f"{i}: {value: .8f} Hartree/e")

Electrostatic potential at QM atoms:
0: -0.00396305 Hartree/e
1: -0.00307742 Hartree/e
2: -0.00941084 Hartree/e
3:  0.00259278 Hartree/e
4: -0.01075681 Hartree/e
5: -0.02470635 Hartree/e
6: -0.00561443 Hartree/e


In [34]:
symbols = [
    "O",
    "H",
    "C",
    "Cl",
    "H",
    "H",
    "H"
]

In [35]:
# Build the QM molecule from your existing initial_xyz
mol = build_molecule(
    np.array([
        [-5.000,  0.000,  0.000],
        [-5.970,  0.000,  0.000],
        [ 0.000,  0.000,  0.000],
        [ 1.780,  0.000,  0.000],
        [-0.630,  0.630,  0.630],
        [-0.630, -0.630,  0.630],
        [-0.630,  0.000, -0.890],
    ])
)

# Classical TIP4P-D charge sites:
# H1, H2, and the M site. O has zero charge.
mm_coords = np.array([
    H1,
    H2,
    M
])

mm_charges = np.array([
    +0.58,
    +0.58,
    -1.16
])

# Add the classical charges to the QM Hamiltonian
mf_qmmm = mm_charge(
    scf.RHF(mol),
    mm_coords,
    mm_charges
)
mf_qmmm.conv_tol = SCF_CONV_TOL
energy_qmmm = mf_qmmm.kernel()

print("QM/MM electrostatic energy:", energy_qmmm)

converged SCF energy = -574.310541464044
QM/MM electrostatic energy: -574.3105414640443


In [36]:
# QM calculation WITHOUT the water charges
mf_qm = scf.RHF(mol)
energy_qm = mf_qm.kernel()

# QM calculation WITH the TIP4P-D charges
mf_qmmm = mm_charge(
    scf.RHF(mol),
    mm_coords,
    mm_charges
)
energy_qmmm = mf_qmmm.kernel()

print(f"QM energy:       {energy_qm:.12f} Hartree")
print(f"QM/MM energy:    {energy_qmmm:.12f} Hartree")
print(f"Difference:      {energy_qmmm - energy_qm:.12f} Hartree")
print(f"Difference:      {(energy_qmmm - energy_qm)*2625.49962:.6f} kJ/mol")

converged SCF energy = -574.308632235187
converged SCF energy = -574.310541464593
QM energy:       -574.308632235187 Hartree
QM/MM energy:    -574.310541464593 Hartree
Difference:      -0.001909229406 Hartree
Difference:      -5.012681 kJ/mol


In [37]:
# This is a test: when the distance between the water's oxygen and the OH- oxygen is sigma, lj energy should be zero
print(lj_energy(
    3.45,
    oh_water_cross["O"]["sigma_A"],
    oh_water_cross["O"]["epsilon_kJmol"]
))

0.0


In [38]:
# Coordinates of the water's oxygen
water_O = O
water_O_A = np.array(water_O)  # already in Angstrom
print("Water O:", water_O_A)

# OH- coordinates
oh_coords = np.array([
    mol.atom_coords()[0],   # O
    mol.atom_coords()[1]    # H
]) * 0.529177  # Bohr -> Angstrom
print("OH- O:", oh_coords[0])
print("OH- H:", oh_coords[1])

# OH- / water-O LJ energy for the current water molecule
r_OO = np.linalg.norm(oh_coords[0] - water_O_A)
E_OO = lj_energy(
    r_OO,
    oh_water_cross["O"]["sigma_A"],
    oh_water_cross["O"]["epsilon_kJmol"]
)
r_HO = np.linalg.norm(oh_coords[1] - water_O_A)
E_HO = lj_energy(
    r_HO,
    oh_water_cross["H"]["sigma_A"],
    oh_water_cross["H"]["epsilon_kJmol"]
)
E_OH_water = E_OO + E_HO

print(f"OH- O ... water O distance: {r_OO:.4f} Å")
print(f"OH- H ... water O distance: {r_HO:.4f} Å")
print(f"O ... water-O LJ energy:     {E_OO:.6f} kJ/mol")
print(f"H ... water-O LJ energy:     {E_HO:.6f} kJ/mol")
print(f"Total OH- / water LJ:        {E_OH_water:.6f} kJ/mol")

Water O: [0. 0. 3.]
OH- O: [-4.99999801  0.          0.        ]
OH- H: [-5.96999762  0.          0.        ]
OH- O ... water O distance: 5.8310 Å
OH- H ... water O distance: 6.6814 Å
O ... water-O LJ energy:     -0.072420 kJ/mol
H ... water-O LJ energy:     -0.002515 kJ/mol
Total OH- / water LJ:        -0.074935 kJ/mol


In [39]:
qm_lj = {
    "OH_O": {
        "sigma_A": 3.400,
        "epsilon_kJmol": 0.2508914038369354
    },

    "OH_H": {
        "sigma_A": 1.443,
        "epsilon_kJmol": 0.18390926154006562
    },

    "C": {
        "sigma_A": 3.39967,
        "epsilon_kJmol": 0.457730
    },

    "Cl": {
        "sigma_A": 4.04468018036,
        "epsilon_kJmol": 0.6276
    },

    "CH3_H": {
        "sigma_A": 2.64953,
        "epsilon_kJmol": 0.0656888
    }
}

# CH3Cl atoms
qm_atom_indices = {
    "C": 2,
    "Cl": 3,
    "H1": 4,
    "H2": 5,
    "H3": 6,
}

# Get QM coordinates in Angstrom
qm_coords_A = mol.atom_coords(unit="Angstrom")

# Water oxygen
water_O_A = np.asarray(O)

# Individual LJ contributions
E_C  = lj_energy(
    np.linalg.norm(qm_coords_A[qm_atom_indices["C"]] - water_O_A),
    qm_lj["C"]["sigma_A"],
    qm_lj["C"]["epsilon_kJmol"]
)

E_Cl = lj_energy(
    np.linalg.norm(qm_coords_A[qm_atom_indices["Cl"]] - water_O_A),
    qm_lj["Cl"]["sigma_A"],
    qm_lj["Cl"]["epsilon_kJmol"]
)

E_H1 = lj_energy(
    np.linalg.norm(qm_coords_A[qm_atom_indices["H1"]] - water_O_A),
    qm_lj["CH3_H"]["sigma_A"],
    qm_lj["CH3_H"]["epsilon_kJmol"]
)

E_H2 = lj_energy(
    np.linalg.norm(qm_coords_A[qm_atom_indices["H2"]] - water_O_A),
    qm_lj["CH3_H"]["sigma_A"],
    qm_lj["CH3_H"]["epsilon_kJmol"]
)

E_H3 = lj_energy(
    np.linalg.norm(qm_coords_A[qm_atom_indices["H3"]] - water_O_A),
    qm_lj["CH3_H"]["sigma_A"],
    qm_lj["CH3_H"]["epsilon_kJmol"]
)

print(f"C  ... water O: {E_C: .6f} kJ/mol")
print(f"Cl ... water O: {E_Cl: .6f} kJ/mol")
print(f"H1 ... water O: {E_H1: .6f} kJ/mol")
print(f"H2 ... water O: {E_H2: .6f} kJ/mol")
print(f"H3 ... water O: {E_H3: .6f} kJ/mol")

E_ch3cl_water = E_C + E_Cl + E_H1 + E_H2 + E_H3

print(f"\nTotal CH3Cl / water LJ: {E_ch3cl_water: .6f} kJ/mol")

C  ... water O:  4.334529 kJ/mol
Cl ... water O:  8.723303 kJ/mol
H1 ... water O:  0.108026 kJ/mol
H2 ... water O:  0.108026 kJ/mol
H3 ... water O: -0.022031 kJ/mol

Total CH3Cl / water LJ:  13.251852 kJ/mol


In [40]:
qm_atom_types = [
    "OH_O",
    "OH_H",
    "C",
    "Cl",
    "CH3_H",
    "CH3_H",
    "CH3_H"
]

In [41]:
water_O_coords_A = np.array([
    O
])

E_LJ = qm_water_lj_energy(
    mol.atom_coords(unit="Angstrom"),
    np.array([O]),
    qm_atom_types
)

print("QM-water LJ:", E_LJ, "kJ/mol")

QM-water LJ: 13.213883127045488 kJ/mol


In [42]:
print("oh_water_cross:")
print(oh_water_cross)

print("\nqm_lj:")
print(qm_lj)

oh_water_cross:
{'O': {'sigma_A': 3.45, 'epsilon_K': 53.0312, 'epsilon_kJmol': 0.44092592998768165}, 'H': {'sigma_A': 2.30095, 'epsilon_K': 45.404, 'epsilon_kJmol': 0.377509860707672}}

qm_lj:
{'OH_O': {'sigma_A': 3.4, 'epsilon_kJmol': 0.2508914038369354}, 'OH_H': {'sigma_A': 1.443, 'epsilon_kJmol': 0.18390926154006562}, 'C': {'sigma_A': 3.39967, 'epsilon_kJmol': 0.45773}, 'Cl': {'sigma_A': 4.04468018036, 'epsilon_kJmol': 0.6276}, 'CH3_H': {'sigma_A': 2.64953, 'epsilon_kJmol': 0.0656888}}


In [43]:
# This is a test
for i, atom_type in enumerate(qm_atom_types):
    sigma = qm_lj[atom_type]["sigma_A"]
    epsilon = qm_lj[atom_type]["epsilon_kJmol"]

    E_atom = 0.0

    for water_O in water_O_coords_A:
        r = np.linalg.norm(qm_coords_A[i] - water_O)

        E_atom += lj_energy(
            r,
            sigma,
            epsilon
        )

    print(f"{i}: {atom_type:2s}  {E_atom: .6f} kJ/mol")

0: OH_O  -0.037894 kJ/mol
1: OH_H  -0.000075 kJ/mol
2: C    4.334529 kJ/mol
3: Cl   8.723303 kJ/mol
4: CH3_H   0.108026 kJ/mol
5: CH3_H   0.108026 kJ/mol
6: CH3_H  -0.022031 kJ/mol


In [44]:
qm_water_cross = {
    "OH_O": {
        "sigma_A": 3.45,
        "epsilon_kJmol": 0.44092592998768165
    },

    "OH_H": {
        "sigma_A": 2.30095,
        "epsilon_kJmol": 0.377509860707672
    },
}

In [45]:
# Current QM geometry
qm_coords_A = mol.atom_coords(unit="Angstrom")

# Current single TIP4P-D water
water_O_coords_A = np.array([O])

# 1. QM/MM electrostatic energy
mf_qmmm = mm_charge(
    scf.RHF(mol),
    mm_coords,
    mm_charges
)
energy_qmmm = mf_qmmm.kernel()

# 2. Bare QM energy
mf_qm = scf.RHF(mol)
energy_qm = mf_qm.kernel()

# 3. QM-water LJ energy
E_LJ = qm_water_lj_energy(
    qm_coords_A,
    water_O_coords_A,
    qm_atom_types
)

# Electrostatic contribution
E_electrostatic_Ha = energy_qmmm - energy_qm
E_electrostatic_kJ = E_electrostatic_Ha * 2625.49962

# Total environmental interaction
E_interaction = E_electrostatic_kJ + E_LJ

print(f"QM energy:                 {energy_qm:.12f} Hartree")
print(f"QM/MM electrostatic:       {E_electrostatic_kJ:.6f} kJ/mol")
print(f"QM-water LJ:               {E_LJ:.6f} kJ/mol")
print(f"Total QM-water interaction:{E_interaction:.6f} kJ/mol")

converged SCF energy = -574.310541464594
converged SCF energy = -574.308632235188
QM energy:                 -574.308632235188 Hartree
QM/MM electrostatic:       -5.012681 kJ/mol
QM-water LJ:               13.213883 kJ/mol
Total QM-water interaction:8.201202 kJ/mol


In [46]:
# Fixed TIP4P-D geometry in a local coordinate system
water_O_local = np.array([0.0, 0.0, 0.0])
water_H1_local = np.array([0.9572, 0.0, 0.0])
water_H2_local = np.array([
    -0.23998617,
     0.92662747,
     0.0
])

water_M_local = np.array([
    0.09462759,
    0.12225716,
    0.0
])

In [47]:
x_test = np.array([
    0.0, 0.0, 3.0,   # O position
    0.0, 0.0, 0.0    # no rotation
])

O_test, H1_test, H2_test, M_test = water_from_variables(x_test)

print("O :", O_test)
print("H1:", H1_test)
print("H2:", H2_test)
print("M :", M_test)

O : [0. 0. 3.]
H1: [0.9572 0.     3.    ]
H2: [-0.23998617  0.92662747  3.        ]
M : [0.09462759 0.12225716 3.        ]


In [48]:
# Initial TIP4P-D water configuration
water_x = np.array([
    0.0, 0.0, 3.0,    # O position
    0.0, 0.0, 0.0     # rotation vector
])

water_result_pre_loop = minimize(
    total_qm_water_energy,
    water_x,
    args=(mol, qm_atom_types, energy_qm),
    method="Powell",
    options={
        "maxiter": 10,
        "xtol": 1e-2,
        "ftol": 1e-2,
        "disp": True
    }
)

converged SCF energy = -574.310541464029
converged SCF energy = -574.310541464029
converged SCF energy = -574.308722242206
converged SCF energy = -574.305196636604
converged SCF energy = -574.309879790603
converged SCF energy = -574.309879790603
converged SCF energy = -574.308900324988
converged SCF energy = -574.30536736237
converged SCF energy = -574.308900324988
converged SCF energy = -574.306633486384
converged SCF energy = -574.389557958973
converged SCF energy = -574.314348797418
converged SCF energy = -574.307553674359
converged SCF energy = -574.308177643746
converged SCF energy = -574.308724040102
converged SCF energy = -574.308558759696
converged SCF energy = -574.308724040103
converged SCF energy = -574.306965388263
converged SCF energy = -574.30482561461
converged SCF energy = -574.309543775052
converged SCF energy = -574.309543775052
converged SCF energy = -574.308308173087
converged SCF energy = -574.307100358162
converged SCF energy = -574.307492082101
converged SCF ener

In [49]:
# Setting up the loop over constrained distances between C and O (of OH-)
ndistances = 1
distances = np.linspace(3.0, 1.2, ndistances)
print("Number of steps:", len(distances))
print("Starting distance:", distances[0])
print("Final distance:", distances[-1])

energies = []
actual_distances = []
optimized_molecules = []
mean_field_objects = []
current_mol = mol
water_coordinates = []
water_energies = []
water_x = np.array([
    0.0, 0.0, 3.0,
    0.0, 0.0, 0.0
])

# Start with no SCF guess.
# After the first calculation, reuse it for subsequent calculations.
previous_dm = None

# Place to put coordinates during the loop
os.makedirs("xyz_frames", exist_ok=True)
for f in glob.glob("xyz_frames/frame_*.xyz"):
    os.remove(f)
print("XYZ folder cleared.")

Number of steps: 1
Starting distance: 3.0
Final distance: 3.0
XYZ folder cleared.


In [50]:
# Loop over constrained distances between C and O (of OH-)
for i, distance in enumerate(distances):

    print("=" * 70)
    print(f"FRAME {i+1}/{ndistances}")
    print(f"Target C-O distance: {distance:.6f} Å")

    # ------------------------------------------------
    # 1. Get previous optimized QM geometry
    # ------------------------------------------------

    current_coords = current_mol.atom_coords(
        unit="Angstrom"
    )

    # ------------------------------------------------
    # 2. Move OH- to the next reaction-coordinate value
    # ------------------------------------------------

    current_coords = move_oh_to_distance(
        current_coords,
        distance
    )

    # ------------------------------------------------
    # 3. Rebuild PySCF molecule
    # ------------------------------------------------

    current_mol = build_molecule(
        current_coords
    )

    # ------------------------------------------------
    # 4. Optimize QM geometry in the electrostatic
    #    field of the CURRENT TIP4P-D water
    # ------------------------------------------------

    current_mol, energy_qm, energy_qmmm = optimize_at_distance(
        current_mol,
        distance,
        water_x
    )

    # ------------------------------------------------
    # 5. Store QM/MM energy
    # ------------------------------------------------

    energy = energy_qmmm

    optimized_molecules.append(current_mol)
    energies.append(energy)

    # ------------------------------------------------
    # 6. Calculate actual C-O distance
    # ------------------------------------------------

    optimized_coords = current_mol.atom_coords(
        unit="Angstrom"
    )

    actual_distance = np.linalg.norm(
        optimized_coords[0] -
        optimized_coords[2]
    )

    actual_distances.append(
        actual_distance
    )

    # ------------------------------------------------
    # 7. Optimize ONE TIP4P-D water molecule
    # ------------------------------------------------

    print("\nOptimizing TIP4P-D water...")

    water_result = minimize(
        total_qm_water_energy,
        water_x,
        args=(
            current_mol,
            qm_atom_types,
            energy_qm
        ),
        method="Powell",
        options={
            "maxiter": 10,
            "xtol": 1e-2,
            "ftol": 1e-2,
            "disp": True
        }
    )

    # ------------------------------------------------
    # 8. Carry optimized water into next frame
    # ------------------------------------------------

    water_x = water_result.x.copy()

    O, H1, H2, M = water_from_variables(
        water_x
    )

    water_coords = np.array([
        O,
        H1,
        H2,
        M
    ])

    water_coordinates.append(
        water_coords.copy()
    )

    water_energies.append(
        water_result.fun
    )

    # ------------------------------------------------
    # 9. Save XYZ
    # ------------------------------------------------

    filename = (
        f"xyz_frames/frame_{i:03d}.xyz"
    )

    write_xyz(
        current_mol,
        filename,
        actual_distance
    )

    # ------------------------------------------------
    # 10. Report
    # ------------------------------------------------

    print(
        f"Actual C-O: "
        f"{actual_distance:.6f} Å"
    )

    print(
        f"QM/MM Energy: "
        f"{energy:.10f} Hartree"
    )

    print(
        f"Water interaction: "
        f"{water_result.fun:.6f} kJ/mol"
    )

    print(
        "Water O position: "
        f"{O[0]:.4f}, "
        f"{O[1]:.4f}, "
        f"{O[2]:.4f} Å"
    )

    print(
        f"Saved: {filename}"
    )

geometric-optimize called with the following command line:
/home/chemistry/venvs/jupyter/lib/python3.14/site-packages/ipykernel_launcher.py -f /home/chemistry/.local/share/jupyter/runtime/kernel-a2618b83-1c8c-484c-b5ef-01a594287f75.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **

FRAME 1/1
Target C-O distance: 3.000000 Å

Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   O  -3.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -3.970000   0.000000   0.000000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
  Cl   1.780000   0.000000   0.000000    0.000000  0.000000  0.000000
   H  -0.630000   0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000  -0.630000   0.630000    0.000000  0.000000  0.000000
   H  -0.630000   0.000000  -0.890000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr



TypeError: 'QMMM_LJ_Gradients' object is not callable

In [ ]:
# Looking at the energies
energies_kj=(energies-energies[0])*2626
plt.figure(figsize=(7,5))
plt.plot(actual_distances,energies_kj,marker="o")
plt.gca().invert_xaxis()
xlabel = "C-O distance (Å)"; plt.xlabel(xlabel)
ylabel = "Energy (kJ/mol)"
plt.ylabel(ylabel)
plt.title("Relaxed C-O Reaction Coordinate")
plt.grid(True)
plt.show()

In [ ]:
test_mol = optimized_molecules[0]

mf_qm = scf.RHF(test_mol)
E_qm = mf_qm.kernel()
print("Bare QM energy:")
print(E_qm, "Hartree")
print(E_qm * au_to_kJ_conversion, "kJ/mol")

E_full = total_qm_water_energy_full(
    water_x,
    test_mol,
    qm_atom_types
)
print("\nFull QM + water energy:")
print(E_full, "kJ/mol")

E_difference = E_full - E_qm*au_to_kJ_conversion
print('Difference = ', E_difference, "kJ/mol")

E_interaction = total_qm_water_energy(
    water_x,
    test_mol,
    qm_atom_types,
    E_qm
)
print("QM-water interaction:", E_interaction, "kJ/mol")

In [ ]:

# Bare QM
mf_qm = scf.RHF(test_mol)
E_qm = mf_qm.kernel()
print("\nBare QM:")
print(E_qm, "Hartree")
print(E_qm*au_to_kJ_conversion, "kJ/mol")

# QM + water
O, H1, H2, M = water_from_variables(water_x)
mm_coords = np.array([H1, H2, M])
mm_charges = np.array([0.58, 0.58, -1.16])
mf_qmmm = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)
E_qmmm = mf_qmmm.kernel()
print("\nQM + water:")
print(E_qmmm, "Hartree")
print(E_qmmm*au_to_kJ_conversion, "kJ/mol")

# LJ
E_LJ = qm_water_lj_energy(
    test_mol.atom_coords(unit="Angstrom"),
    np.array([O]),
    qm_atom_types
)
print("\nQM/MM, lj contribution:")
print(E_LJ, "kJ/mol")

E_electrostatic = (E_qmmm - E_qm) *au_to_kJ_conversion
print("\nQM/MM, electrostatic contribution:")
print(
    E_electrostatic,
    "kJ/mol"
)

E_total_interaction = E_electrostatic + E_LJ
print("\nQM/MM, total interaction:")
print(
    E_total_interaction,
    "kJ/mol"
)

E_full = E_qmmm *au_to_kJ_conversion + E_LJ
print("\nFull energy:")
print(
    E_full,
    "kJ/mol"
)

print("\nFull - bare:")
print(
    E_qmmm *au_to_kJ_conversion
    + E_LJ
    - E_qm *au_to_kJ_conversion,
    "kJ/mol"
)

In [ ]:
for i, water in enumerate(water_coordinates):
    print(f"\nFrame {i}:")
    print(f"O : {water[0]}")
    print(f"H1: {water[1]}")
    print(f"H2: {water[2]}")
    print(f"M : {water[3]}")

In [ ]:
test_mol = optimized_molecules[0]
water_x = np.array([
    0.0, 0.0, 3.0,
    0.0, 0.0, 0.0
])

water_result = minimize(
    total_qm_water_energy,
    water_x,
    args=(
        test_mol,
        qm_atom_types,
        E_qm
    ),
    method="Powell",
    bounds=[
        (-6.0, 6.0),          # Ox
        (-6.0, 6.0),          # Oy
        (-6.0, 6.0),          # Oz
        (-np.pi, np.pi),      # rx
        (-np.pi, np.pi),      # ry
        (-np.pi, np.pi)       # rz
    ],
    options={
        "maxiter": 20,
        "xtol": 1e-3,
        "ftol": 1e-3,
        "disp": True
    }
)

print("\nOptimization result:")
print(water_result)

print("\nOptimized water variables:")
print(water_result.x)

O, H1, H2, M = water_from_variables(water_result.x)

print("\nOptimized water:")
print("O :", O)
print("H1:", H1)
print("H2:", H2)
print("M :", M)

print("\nWater interaction:")
print(water_result.fun, "kJ/mol")

In [ ]:
test_water_x = water_result.x.copy()

mol_test, E_qm_test, E_qmmm_test = optimize_at_distance(
    test_mol,
    distances[0],
    test_water_x
)

print("\nRESULT")

print("Bare QM energy:")
print(E_qm_test, "Hartree")

print("\nQM/MM energy:")
print(E_qmmm_test, "Hartree")

print("\nElectrostatic interaction:")
print(
    (E_qmmm_test - E_qm_test)
    * au_to_kJ_conversion,
    "kJ/mol"
)

print("\nC-O distance:")

coords = mol_test.atom_coords(unit="Angstrom")

print(
    np.linalg.norm(coords[0] - coords[2]),
    "Å"
)

In [ ]:
O, H1, H2, M = water_from_variables(test_water_x)

print("\nWater:")
print("O :", O)
print("H1:", H1)
print("H2:", H2)
print("M :", M)

print("\nOptimized QM coordinates:")
print(mol_test.atom_coords(unit="Angstrom"))


qm_coords = mol_test.atom_coords(unit="Angstrom")

print("\nDistances from water O to QM atoms:")

for i, coord in enumerate(qm_coords):
    r = np.linalg.norm(coord - O)
    print(f"QM atom {i}: {r:.4f} Å")

E_LJ = qm_water_lj_energy(
    mol_test.atom_coords(unit="Angstrom"),
    np.array([O]),
    qm_atom_types
)

print("\nLJ interaction:")
print(E_LJ, "kJ/mol")

In [ ]:
O, H1, H2, M = water_from_variables(test_water_x)

qm_coords = mol_test.atom_coords(
    unit="Angstrom"
)

E_LJ_old = qm_water_lj_energy(
    qm_coords,
    np.array([O]),
    qm_atom_types
)

E_LJ_new, grad_LJ = qm_water_lj_energy_gradient(
    qm_coords,
    O,
    qm_atom_types
)

print("Old LJ energy:", E_LJ_old)
print("New LJ energy:", E_LJ_new)

print("\nLJ gradient:")
print(grad_LJ)

In [27]:
O, H1, H2, M = water_from_variables(water_x)

mm_coords = np.array([H1, H2, M])
mm_charges = np.array([0.58, 0.58, -1.16])

mf = mm_charge(
    scf.RHF(test_mol),
    mm_coords,
    mm_charges,
    unit="Angstrom"
)

mf.kernel()

grad = QMMM_LJ_Gradients(
    mf,
    O,
    qm_atom_types
)

g = grad.kernel()

print("Combined gradient:")
print(g)

print("\nGradient shape:")
print(g.shape)

print("\nMaximum absolute gradient:")
print(np.max(np.abs(g)))

NameError: name 'test_mol' is not defined

In [ ]:
qm_coords_A = test_mol.atom_coords(unit="Angstrom")
water_x = np.array([
    2.5, 0.0, 4.0,
    0.0, 0.0, 0.0
])
O, H1, H2, M = water_from_variables(water_x)

print("Water O:", O)

print("\nDistances from water O to QM atoms:")

for i, coord in enumerate(qm_coords_A):
    r = np.linalg.norm(coord - O)
    print(
        f"QM atom {i} ({test_mol.atom_symbol(i)}): "
        f"{r:.8f} Å"
    )


In [ ]:
water_x = np.array([
    2.5, 0.0, 4.0,
    0.0, 0.0, 0.0
])

O, H1, H2, M = water_from_variables(water_x)

E_total = total_qm_water_energy(
    water_x,
    test_mol,
    qm_atom_types,
    E_qm
)

print("Initial QM-water interaction:")
print(E_total, "kJ/mol")

E_LJ, grad_LJ = qm_water_lj_energy_gradient(
    qm_coords_A,
    O,
    qm_atom_types
)

print("\nLJ energy:")
print(E_LJ, "kJ/mol")

print("\nMaximum LJ gradient:")
print(
    np.max(np.abs(grad_LJ)),
    "kJ/mol/Å"
)

water_result = minimize(
    total_qm_water_energy,
    water_x,
    args=(
        test_mol,
        qm_atom_types,
        E_qm
    ),
    method="Powell",
    options={
        "maxiter": 10,
        "xtol": 1e-2,
        "ftol": 1e-2,
        "disp": True
    }
)

water_x_opt = water_result.x.copy()

O, H1, H2, M = water_from_variables(
    water_x_opt
)

print("\nOptimized water:")
print("O :", O)
print("H1:", H1)
print("H2:", H2)
print("M :", M)

print(
    "\nInteraction:",
    water_result.fun,
    "kJ/mol"
)

In [28]:
grad = QMMM_LJ_Gradients(
    mf,
    O,
    qm_atom_types
)

g = grad.kernel()

print("Gradient shape:", g.shape)
print("Maximum absolute gradient:", np.max(np.abs(g)))
print(g)

NameError: name 'mf' is not defined